In [ ]:
from networkx.convert import to_networkx_graph
from torch_geometric.utils import to_networkx

import AttackerGNN.MainAttack
import config
import os

import numpy as np

from matplotlib import pyplot as plt

from experiments import (
    experiment_train,
    experiment_local_attack_direct,
    experiment_global_attack_direct
)
import helpers.selector_pipeline_helpers

from sparse_smoothing.utils import load_and_standardize
from sparse_smoothing.models import GCN

%cd {config.PROJECT_DIR}

%load_ext autoreload
%autoreload 2

files = ["cache/demo.json", "cache/demo/demo_1.pt", "cache/evasion_global_adj.json", "cache/evasion_global_attr.json", "cache/evasion_global_adj/evasion_global_adj_1.pt", "cache/evasion_global_attr/evasion_global_attr_1.pt"]

for file_path in files:
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"{file_path} has been deleted.")
    else:
        print(f"{file_path} does not exist.")

%matplotlib inline

In [ ]:
# ── Dataset / split ─────────────────────────────────────────────────────────
DATASET = 'cora_ml'
SAMPLING = 'stratified'
SPLIT = 'ratio'
SEED = 0
TRAIN_RATIO = 0.15
VAL_RATIO = 0.10
TEST_RATIO = 0.30

# ── Victim model ─────────────────────────────────────────────────────────────
MODEL_NAME = 'GCN'
MODEL_LABEL = 'GCN'
TRAIN_EPOCHS = 200
HIDDEN = 64
INDUCTIVE = True
DROPOUT_VICTIM = 0.5
LR_VICTIM = 1e-2
WEIGHT_DECAY_VICTIM = 1e-3
PATIENCE_VICTIM = 300
MAX_EPOCHS_VICTIM = 3000

# ── V4 mining params ─────────────────────────────────────────────────────────
USE_PRBCD_CANDIDATES = False
N_PRBCD_RUNS = 3
MINER = 'lrbcd'  # 'lrbcd' or 'prbcd'
MINER_LOCAL_FACTOR = 0.5
DEGREE_CAP_PCT = None  # e.g. 0.9 to cap top-10% hubs
PRBCD_BUDGET_FRACTION = 0.5
PRBCD_EPOCHS = 5
PRBCD_SEARCH_SPACE = 200_000
PRBCD_N_EPOCHS_RESAMPLING = 1

# ── V4 subset-scoring params ─────────────────────────────────────────────────
N_SUBSETS = 1000
SUBSET_FRACTION = 0.05  # each subset = round(fraction × n_candidates)

# ── Scorer training params ───────────────────────────────────────────────────
SCORER_HIDDEN = 64
SCORER_EPOCHS = 200
SCORER_LR = 0.01
VAL_FRACTION = 0.2
PATIENCE = 20
BALANCE_RATIO = 1.0  # neg:pos ratio for zero-label undersampling

In [ ]:
import torch

train_statistics = experiment_train.run(
    data_dir='./data',
    dataset=DATASET,
    model_params=dict(
        label=MODEL_LABEL,
        model=MODEL_NAME,
        do_cache_adj_prep=True,
        n_filters=64,
        dropout=DROPOUT_VICTIM,
        svd_params=None,
        jaccard_params=None,
        gdc_params={"alpha": 0.15, "k": 64},
    ),
    train_params=dict(
        lr=LR_VICTIM,
        weight_decay=WEIGHT_DECAY_VICTIM,
        patience=PATIENCE_VICTIM,
        max_epochs=MAX_EPOCHS_VICTIM,
    ),
    binary_attr=False,
    make_undirected=True,
    seed=SEED,
    artifact_dir='cache',
    model_storage_type='demo_custom_split',
    ppr_cache_params=dict(),
    device="cpu",
    data_device="cpu",
    display_steps=100,
    debug_level="info",
    custom_split_ratios=(TRAIN_RATIO, VAL_RATIO, TEST_RATIO),
)

# plot train and val loss curves
fig, ax = plt.subplots()

color = plt.rcParams['axes.prop_cycle'].by_key()['color'][0]
ax.set_xlabel('Epoch $t$')
ax.set_ylabel("Loss")
ax.plot(train_statistics['trace_train'], color=color, label='Train')

color = plt.rcParams['axes.prop_cycle'].by_key()['color'][1]
ax.plot(train_statistics['trace_val'], color=color, label='Val')
ax.legend()
plt.show()

clean_acc = train_statistics["accuracy"]
print(f'Accuracy of the model: {100 * clean_acc:.2f}%')

model = train_statistics["model"]
graph = train_statistics["graph"]
idx_train = train_statistics["idx_train"]
idx_val = train_statistics["idx_val"]
idx_test = train_statistics["idx_test"]
model.eval()

In [ ]:
import torch

attr_matrix, adj_matrix, labels_raw = graph[:3]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Display ────────────────────────────────────────────────────────────────
n_nodes = adj_matrix.sizes()[0]
n_edges_directed = adj_matrix.nnz()
n_undirected = n_edges_directed // 2

print(f"nodes={n_nodes}")
print(f"Directed edges: {n_edges_directed}")
print(f"Undirected edges: {n_undirected}")
print(f"features={attr_matrix.shape}")
print(f"classes={len(torch.unique(labels_raw))}")

# ── Convert adjacency SparseTensor -> edge_index / edge_weight ─────────────
row, col, value = adj_matrix.coo()

edge_index = torch.stack([row, col], dim=0).long().to(device)

if value is None:
    edge_weight = torch.ones(edge_index.size(1), dtype=torch.float32, device=device)
else:
    edge_weight = value.float().to(device)

# ── Convert features / labels ──────────────────────────────────────────────
attr = attr_matrix.float().to(device)
labels = labels_raw.long().to(device)

# ── Safety checks ──────────────────────────────────────────────────────────
print("edge_index shape:", edge_index.shape)
print("edge_weight shape:", edge_weight.shape)
print("max node id in edge_index:", int(edge_index.max()))
print("n_nodes:", n_nodes)

assert int(edge_index.max()) < n_nodes
assert attr.shape[0] == n_nodes
assert labels.shape[0] == n_nodes
'''
train_idx_np, val_idx_np, test_idx_np = train_val_test_split(
    graph.labels,
    train_size=0.05,
    val_size=0.05,
    test_size=0.9,
    mode="stratified",
    seed=SEED,
)

train_idx = torch.as_tensor(train_idx_np, dtype=torch.long, device=device)
val_idx = torch.as_tensor(val_idx_np, dtype=torch.long, device=device)
test_idx = torch.as_tensor(test_idx_np, dtype=torch.long, device=device)'''

In [ ]:
from helpers.selector_pipeline_helpers import EndpointPRBCDV4Scorer, _edge_set
import random

# ── Candidate sampling ──────────────────────────────────────────────────────
budget = max(1, round(PRBCD_BUDGET_FRACTION * n_undirected))

if USE_PRBCD_CANDIDATES:
    # ── Run endpointPRBCD/V4-style scorer ───────────────────────────────────
    scorer = EndpointPRBCDV4Scorer(
        n=n_nodes,
        edge_index=edge_index,
        edge_weight=edge_weight,
        attr=attr,
        labels=labels,
        attacked_model=model,
        idx_attack=idx_test,
        test_idx=idx_val,
        device=device,
        block_size=PRBCD_SEARCH_SPACE,
        n_epochs_resampling=PRBCD_N_EPOCHS_RESAMPLING,

        n_subsets=N_SUBSETS,
        subset_fraction=SUBSET_FRACTION,
        balance_ratio=BALANCE_RATIO,
        store_candidates=True,
    )

    candidates = scorer.mine_prbcd_endpoint_candidates(
        n_perturbations=budget,
        tag="endpointPRBCD",
        rng_seed=SEED,
        n_candidates=PRBCD_SEARCH_SPACE,
        prbcd_epochs=PRBCD_EPOCHS,
        n_prbcd_runs=N_PRBCD_RUNS,
        n_epochs_resampling=PRBCD_N_EPOCHS_RESAMPLING,
    )

else:
    # same candidate count as PRBCD budget over all runs:
    # num_edges * budget_fraction * n_prbcd_runs
    n_random_candidates = max(
        1,
        round(n_undirected * PRBCD_BUDGET_FRACTION * N_PRBCD_RUNS)
    )

    rng = random.Random(SEED)
    candidates_set = set()

    while len(candidates_set) < n_random_candidates:
        u = rng.randrange(n_nodes)
        v = rng.randrange(n_nodes)

        if u == v:
            continue

        candidates_set.add((min(u, v), max(u, v)))

    candidates = list(candidates_set)

# ── Inspect mined / sampled candidates ──────────────────────────────────────
print(f"Candidates: {len(candidates)}")

orig_edges = _edge_set(edge_index)
n_cands    = len(candidates)
n_removals = sum(1 for e in candidates if e in orig_edges)
n_adds     = n_cands - n_removals

print(f"Candidates: {n_cands}  ({n_adds} additions, {n_removals} removals)")
print(f"Original edges: {orig_edges}")